# 11 - Pest Symptom Classifier (YOLOv8 Classification Mode)

Stage 11: Pest symptom classifier (YOLOv8 classification mode).

IMPORTANT SCOPE NOTE: this is a CLASSIFIER, not an object detector. Your
pest folders (Aphids, Army worm, Leaf Miner(s), Spider Mite, Thrips) are
plain folder-per-class images with no bounding box annotations, so true
YOLO object detection (drawing a box around the insect / damage) isn't
possible with this data as-is. This trains YOLOv8's classification head
instead -- same "what's in this photo" task as your disease classifiers,
just using the pest-symptom folders you set aside earlier.

If you want true bounding-box pest detection later, you'll need to
annotate a subset of these images (Roboflow/CVAT/LabelImg) or bring in a
pre-annotated dataset (e.g. IP102) -- that's a separate, bigger effort.

Pest classes are combined ACROSS crops (e.g. "Aphids" from Cotton +
"Aphid" from Pepper Bell become one "Aphid" class) since the same pest
looks the same regardless of which crop it's on, and individual
per-crop counts are too small to train on alone.

Install deps:
    pip install ultralytics pandas scikit-learn tqdm --break-system-packages

## Imports & Configuration

In [ ]:
import shutil
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from ultralytics import YOLO

MANIFEST_PATH = Path("manifest.csv")
OUT_DIR = Path("pest_dataset")
MIN_CLASS_COUNT = 15  # below this, a class is dropped rather than trained on

# Raw folder name -> canonical pest name, combined across all crops.
PEST_CANONICAL_MAP = {
    "Aphids": "Aphid",
    "Aphid": "Aphid",
    "Army worm": "Army Worm",
    "Leaf Miner": "Leaf Miner",
    "Leaf miners": "Leaf Miner",
    "Leaf miner": "Leaf Miner",
    "Spider mite": "Spider Mite",
    "Spider Mite": "Spider Mite",
    "Thrips": "Thrips",  # only ~3 images total -- almost certainly gets pruned below
}

## `build_pest_manifest`

In [ ]:
def build_pest_manifest():
    df = pd.read_csv(MANIFEST_PATH)
    df = df[df["disease_raw"].isin(PEST_CANONICAL_MAP.keys())].copy()
    df["pest"] = df["disease_raw"].map(PEST_CANONICAL_MAP)
    print("Combined pest class counts (across all crops):")
    print(df.groupby("pest").size().sort_values(ascending=False))
    return df

## `prune_rare_classes`

In [ ]:
def prune_rare_classes(df, min_count=MIN_CLASS_COUNT):
    counts = df["pest"].value_counts()
    rare = counts[counts < min_count].index.tolist()
    if rare:
        print(f"\nDropping rare classes (< {min_count} images): {rare}")
        print("These don't have enough images to train on reliably -- collect more if you need them.")
    return df[~df["pest"].isin(rare)]

## `split_and_materialize`

In [ ]:
def split_and_materialize(df, train_size=0.7, val_size=0.15, test_size=0.15, seed=42):
    strat_key = df["pest"]
    train_df, temp_df = train_test_split(df, train_size=train_size, stratify=strat_key, random_state=seed)
    remaining_key = temp_df["pest"]
    relative_val = val_size / (val_size + test_size)
    val_df, test_df = train_test_split(temp_df, train_size=relative_val, stratify=remaining_key, random_state=seed)

    for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
        for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc=f"Copying {split_name}"):
            dest_dir = OUT_DIR / split_name / row["pest"].replace(" ", "_")
            dest_dir.mkdir(parents=True, exist_ok=True)
            src = Path(row["filepath"])
            dest = dest_dir / f"{src.stem}_{abs(hash(str(src)))}{src.suffix}"
            shutil.copy2(src, dest)

    print(f"\nTrain: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")

## `train_pest_classifier`

Ultralytics classification mode expects OUT_DIR to contain train/ and

In [ ]:
def train_pest_classifier(epochs=30, imgsz=224):
    """
    Ultralytics classification mode expects OUT_DIR to contain train/ and
    val/ subfolders of class-named directories -- exactly what
    split_and_materialize() produced above. No data.yaml needed for
    classification (unlike detection mode).
    """
    model = YOLO("yolov8n-cls.pt")  # nano classification model -- fast, good starting point
    results = model.train(
        data=str(OUT_DIR.resolve()),
        epochs=epochs,
        imgsz=imgsz,
        project="pest_classifier_runs",
        name="yolov8n_cls_pest",
    )
    return model, results

## `evaluate_on_test`

In [ ]:
def evaluate_on_test(model):
    test_dir = OUT_DIR / "test"
    metrics = model.val(data=str(OUT_DIR.resolve()), split="test")
    print(f"\nTop-1 accuracy: {metrics.top1:.4f}")
    print(f"Top-5 accuracy: {metrics.top5:.4f}")
    return metrics

## Run

In [ ]:
df = build_pest_manifest()
df = prune_rare_classes(df)
split_and_materialize(df)

model, train_results = train_pest_classifier()
evaluate_on_test(model)

print(f"\nBest weights saved under: pest_classifier_runs/yolov8n_cls_pest/weights/best.pt")